# Figure 2 — learning single-branch evolution

Reproduces the panels of this figure. Each cell runs a released producer and shows its output.

**Default level is 1** — redraw from a table shipped in the Zenodo deposit. No model, no GPU;
seconds on a laptop. Cells that need a GPU or a checkpoint are marked and left commented out.

Run from the `peint-paper` repository, or set `PEINT_PAPER_REPO` to point at it.

In [ ]:
import os, sys, subprocess, pathlib
# Point these at your checkout. PAPER_REPO is the peint-paper repository; PEINT_REPO the model
# library. Everything below runs from the paper repository root.
PAPER_REPO = pathlib.Path(os.environ.get("PEINT_PAPER_REPO", "..")).resolve()
os.environ.setdefault("PEINT_PAPER_PEINT_REPO", str(PAPER_REPO.parent / "peint"))
os.environ["PEINT_PAPER_LOCAL_DATA_ONLY"] = "1"
os.chdir(PAPER_REPO)
sys.path.insert(0, str(PAPER_REPO))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Inline display needs IPython. These notebooks are also executed headlessly in CI, where it is
# absent, so fall back to printing rather than failing the whole notebook on the import.
try:
    from IPython.display import display, Image
    _INLINE = True
except ImportError:
    _INLINE = False
    def display(x): print(x)
    def Image(filename=None, width=None): return f"[figure written to {filename}]"

def run(module, *args):
    """Run a panel producer and show its output."""
    cmd = [sys.executable, "-m", module, *args]
    print("$", " ".join(cmd[2:]))
    r = subprocess.run(cmd, capture_output=True, text=True)
    tail = [l for l in r.stdout.split("\n") if l.strip()][-15:]
    print("\n".join(tail))
    if r.returncode:
        print(f"\n  *** FAILED (exit {r.returncode}) ***")
        print("  " + r.stderr.strip().split("\n")[-1][:400])
    return r.returncode == 0

def show(*names, w=760):
    """Display panel images from figures/output."""
    for n in names:
        p = PAPER_REPO / "figures" / "output" / (n + ".png")
        if p.exists():
            display(Image(filename=str(p), width=w))
        elif (p.with_suffix(".pdf")).exists():
            print(f"  {n}.pdf written (no PNG to display inline)")
        else:
            print("  not found:", n)

def compare(label, ours, printed, tol=0.05):
    """Print our value against the manuscript's."""
    ok = "MATCH" if abs(ours - printed) <= tol else "CHECK"
    print(f"  {label:<28} ours {ours:>8.3f}   manuscript {printed:>8.3f}   {ok}")

## 2a — held-out likelihood vs evolutionary time

Level 1 redraws from the shipped per-`(model, t_bin)` table. Level 3 rescoring the
transitions takes about six hours on one A100 for the published 553+553 split.

In [ ]:
run("figures.figure2_ll_eval_esmc", "--from-csv")
show("figure2_likelihood_eval_test_esmc", "figure2_likelihood_eval_train_held_out_esmc")

## 2b — evolutionary time estimation

Level 3 only, and **use the default 150 families**: a smaller subsample completes but
produces a near-empty-looking figure, because the density plot is scaled for the full run.

In [ ]:
# Requires a GPU and a checkpoint:
# run("figures.figure2_time_estimation", "--checkpoint", "<ckpt>",
#     "--num-families", "150", "--batch-size", "32")
show("figure2_time_estimation_all")

## 2c — evolutionary events and pLDDT vs time

Two stages in two environments: generation, then folding.

In [ ]:
# run("figures.figure2_simulation", "--family", "3t0y_1_A", "--skip-structures")
show("figure2_simulation_mutations", "figure2_simulation_plddt")